# LeetCode 840 — Magic Squares In Grid bằng Python

Link đề: https://leetcode.com/problems/magic-squares-in-grid/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Math, Matrix

Skeleton LeetCode:

```python
class Solution:
    def numMagicSquaresInside(self, grid: List[List[int]]) -> int:
        ...
```

Ghi chú: bài có **đáp án duy nhất** (một số nguyên). Điểm thú vị: chỉ có **đúng 8** ma phương `3 × 3` dùng các số `1..9` (một ma phương gốc cùng 4 phép xoay và phép lật), và notebook **chứng minh bằng máy** điều này bằng cách duyệt cả `9! = 362 880` hoán vị.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số ô vuông con `3 × 3` trong lưới `row × col` là **ma phương**: gồm các số **phân biệt từ `1` đến `9`** và **mọi hàng, mọi cột, cả hai đường chéo** có cùng tổng.
2. Nắm các công cụ Python: duyệt mọi cửa sổ `3 × 3` bằng hai vòng `for`, cắt lát `row[j:j+3]`, `set`, `sum`, `zip(*block)` (cột), `itertools.permutations`, mã hoá cửa sổ thành bộ `tuple`/số nguyên để tra trong `set`.
3. Đi từ **kiểm tra định nghĩa cho từng cửa sổ (brute force)** → **lọc nhanh bằng tính chất toán học (tâm bằng `5`)** → **tra cứu 8 ma phương có sẵn** → **so khớp "vòng ngoài" (ring) với mẫu `4 9 2 7 6 1 8 3`**.
4. Hiểu mấu chốt: tổng mọi số `1..9` là `45` nên mỗi hàng phải có tổng `15`; cộng bốn đường qua tâm (hàng giữa, cột giữa, hai đường chéo) cho `60 = 45 + 3·tâm`, suy ra **tâm `= 5`**; từ đó chỉ còn 8 ma phương.
5. Tránh các lỗi kinh điển: bỏ sót **đường chéo**, quên điều kiện **phân biệt** hoặc **trong `1..9`**, nhầm lẫn "tổng bằng nhau" với "ma phương" (ma phương **bán phần** — semi-magic), lệch chỉ số khi trượt cửa sổ (`range(row − 2)`), và tin rằng "tâm bằng 5" là đủ.

## 2. Đề bài & ví dụ

Một **ma phương `3 × 3`** là lưới `3 × 3` điền các số **phân biệt từ `1` đến `9`** sao cho **mỗi hàng, mỗi cột và cả hai đường chéo** đều có cùng tổng.

Cho lưới số nguyên `grid` kích thước `row × col`. Hỏi có bao nhiêu **lưới con** `3 × 3` (các ô liên tiếp tạo thành ma phương)?

Chú ý: một ma phương chỉ chứa các số từ `1` đến `9`, nhưng `grid` có thể chứa các số tới `15`.

**Ví dụ 1:** `grid = [[4,3,8,4],[9,5,1,9],[2,7,6,2]]` → `1`. Lưới con (cột `0..2`) `[[4,3,8],[9,5,1],[2,7,6]]` là ma phương (mọi đường tổng `15`); lưới con (cột `1..3`) `[[3,8,4],[5,1,9],[7,6,2]]` thì không.

**Ví dụ 2:** `grid = [[8]]` → `0`.

**Ràng buộc:** `row == grid.length`, `col == grid[i].length`; `1 <= row, col <= 10`; `0 <= grid[i][j] <= 15`.

In [1]:
import itertools
import random
import time
from itertools import permutations
from typing import List

ex1 = [[4, 3, 8, 4], [9, 5, 1, 9], [2, 7, 6, 2]]
ex2 = [[8]]
for row in ex1:
    print(row)
print("kích thước:", len(ex1), "x", len(ex1[0]), "-> số cửa sổ 3x3:", (len(ex1) - 2) * (len(ex1[0]) - 2))

[4, 3, 8, 4]
[9, 5, 1, 9]
[2, 7, 6, 2]
kích thước: 3 x 4 -> số cửa sổ 3x3: 2


## 3. Kiến thức nền

### 3.1 Duyệt mọi cửa sổ `3 × 3`

Lưới `row × col` có `(row − 2) · (col − 2)` cửa sổ `3 × 3` (góc trên trái ở `(i, j)` với `0 ≤ i ≤ row − 3`, `0 ≤ j ≤ col − 3`). Nếu `row < 3` hoặc `col < 3` thì **không có** cửa sổ nào (kết quả `0`). Hai vòng `for i in range(row − 2)`, `for j in range(col − 2)` duyệt đúng các cửa sổ; `range` của số âm hoặc `0` là rỗng nên không cần xử lý riêng.

In [2]:
def windows(grid):
    m, n = len(grid), len(grid[0])
    for i in range(m - 2):
        for j in range(n - 2):
            yield i, j, [grid[i + k][j:j + 3] for k in range(3)]


for i, j, block in windows(ex1):
    print((i, j), block)
print("[[8]] có", len(list(windows([[8]]))), "cửa sổ; 2x5 có", len(list(windows([[0] * 5] * 2))), "cửa sổ")

(0, 0) [[4, 3, 8], [9, 5, 1], [2, 7, 6]]
(0, 1) [[3, 8, 4], [5, 1, 9], [7, 6, 2]]
[[8]] có 0 cửa sổ; 2x5 có 0 cửa sổ


### 3.2 Cắt lát (slicing), cột bằng `zip(*block)` và tổng

`row[j:j + 3]` là 3 phần tử liên tiếp. Từ khối `block` (list gồm 3 hàng) có thể lấy **cột** bằng `zip(*block)` (chuyển vị). Tổng mỗi hàng `sum(r)`, mỗi cột `sum(c)`; đường chéo chính `block[0][0] + block[1][1] + block[2][2]`, đường chéo phụ `block[0][2] + block[1][1] + block[2][0]`.

In [3]:
block = [[4, 3, 8], [9, 5, 1], [2, 7, 6]]
print("hàng :", [sum(r) for r in block])
print("cột  :", [sum(c) for c in zip(*block)])
print("chéo :", block[0][0] + block[1][1] + block[2][2], block[0][2] + block[1][1] + block[2][0])

hàng : [15, 15, 15]
cột  : [15, 15, 15]
chéo : 15 15


### 3.3 `set` để kiểm tra "phân biệt và đủ `1..9`"

`set(values)` bỏ các phần tử trùng. Với 9 số, `set(values) == set(range(1, 10))` đúng khi và chỉ khi 9 số là **hoán vị của `1..9`** (phân biệt, đủ chín giá trị, đều trong `[1, 9]`). Đây là một kiểm tra gọn thay cho ba điều kiện riêng (phân biệt, `≥ 1`, `≤ 9`).

In [4]:
flat = [4, 3, 8, 9, 5, 1, 2, 7, 6]
print(set(flat) == set(range(1, 10)), "| trùng:", set([5] * 9) == set(range(1, 10)), "| vượt 9:", set([1, 2, 3, 4, 5, 6, 7, 8, 10]) == set(range(1, 10)))
print("sorted(flat) == list(range(1, 10)):", sorted(flat) == list(range(1, 10)))

True | trùng: False | vượt 9: False
sorted(flat) == list(range(1, 10)): True


### 3.4 Tổng 45 và vì sao tổng mỗi hàng là 15

Tổng `1 + 2 + … + 9 = 45`. Ba hàng chia đều `45` nên **tổng mỗi hàng `= 15`** (tương tự cột, đường chéo). Đây là cách biết ngay tổng cần so sánh mà không phải tính hàng đầu tiên.

### 3.5 Hoán vị `itertools.permutations`

`permutations(range(1, 10))` sinh `9! = 362 880` hoán vị. Mỗi hoán vị đọc thành lưới `3 × 3` (theo hàng). Ta dùng nó để **vét cạn mọi cách điền `1..9`** và đếm bao nhiêu cách là ma phương.

In [5]:
def is_magic_brute(block):
    flat = [v for r in block for v in r]
    if sorted(flat) != list(range(1, 10)):
        return False
    s = sum(block[0])
    return (all(sum(r) == s for r in block)
            and all(sum(c) == s for c in zip(*block))
            and block[0][0] + block[1][1] + block[2][2] == s
            and block[0][2] + block[1][1] + block[2][0] == s)


t0 = time.perf_counter()
magic_squares = [p for p in permutations(range(1, 10)) if is_magic_brute([p[0:3], p[3:6], p[6:9]])]
print("duyệt", 362880, "hoán vị trong", round(time.perf_counter() - t0, 2), "s; số ma phương 3x3:", len(magic_squares))
for p in magic_squares:
    print(p[0:3], p[3:6], p[6:9])
assert len(magic_squares) == 8

duyệt 362880 hoán vị trong 0.54 s; số ma phương 3x3: 8
(2, 7, 6) (9, 5, 1) (4, 3, 8)
(2, 9, 4) (7, 5, 3) (6, 1, 8)
(4, 3, 8) (9, 5, 1) (2, 7, 6)
(4, 9, 2) (3, 5, 7) (8, 1, 6)
(6, 1, 8) (7, 5, 3) (2, 9, 4)
(6, 7, 2) (1, 5, 9) (8, 3, 4)
(8, 1, 6) (3, 5, 7) (4, 9, 2)
(8, 3, 4) (1, 5, 9) (6, 7, 2)


Vậy **đúng 8** ma phương `3 × 3`. Mọi cách giải thông minh bên dưới thực chất chỉ tận dụng sự thật này.

## 4. Ý tưởng nền tảng / trực giác

Cách đơn giản nhất: với mỗi cửa sổ `3 × 3`, **kiểm tra định nghĩa**:

1. Chín số phải là `1..9` (mỗi số đúng một lần).
2. Ba hàng, ba cột, hai đường chéo đều có tổng `15`.

Tính tay ví dụ 1: cửa sổ `(0, 0)` = `[[4,3,8],[9,5,1],[2,7,6]]`:

- chín số `{4,3,8,9,5,1,2,7,6}` = `{1..9}` ✓.
- hàng: `15, 15, 15`; cột: `4+9+2 = 15`, `3+5+7 = 15`, `8+1+6 = 15`; đường chéo: `4+5+6 = 15`, `8+5+2 = 15` ✓ → ma phương.

Cửa sổ `(0, 1)` = `[[3,8,4],[5,1,9],[7,6,2]]`: chín số vẫn là `1..9`, nhưng hàng `3+8+4 = 15`, `5+1+9 = 15`, `7+6+2 = 15`; cột `3+5+7 = 15`, `8+1+6 = 15`, `4+9+2 = 15`; đường chéo `3+1+2 = 6 ≠ 15` ✗ → không phải ma phương (đây là một **ma phương bán phần** — hàng và cột đúng, đường chéo sai).

**Quan sát về tâm:** ở cả hai ma phương hợp lệ ta thấy ô giữa là `5`; ở cửa sổ sai ô giữa là `1`. Có phải mọi ma phương `3 × 3` đều có tâm `5`? Có — chứng minh ở mục 6.

In [6]:
for i, j, block in windows(ex1):
    flat = [v for r in block for v in r]
    print(f"cửa sổ {(i, j)}: {block}")
    print("   là hoán vị 1..9 :", sorted(flat) == list(range(1, 10)))
    print("   tổng hàng/cột/chéo:", [sum(r) for r in block], [sum(c) for c in zip(*block)],
          block[0][0] + block[1][1] + block[2][2], block[0][2] + block[1][1] + block[2][0])
    print("   => ma phương?", is_magic_brute(block))

cửa sổ (0, 0): [[4, 3, 8], [9, 5, 1], [2, 7, 6]]
   là hoán vị 1..9 : True
   tổng hàng/cột/chéo: [15, 15, 15] [15, 15, 15] 15 15
   => ma phương? True
cửa sổ (0, 1): [[3, 8, 4], [5, 1, 9], [7, 6, 2]]
   là hoán vị 1..9 : True
   tổng hàng/cột/chéo: [15, 15, 15] [15, 15, 15] 6 12
   => ma phương? False


## 5. Cách 1 — Brute Force: kiểm tra định nghĩa cho từng cửa sổ

**Ý tưởng:** duyệt mọi cửa sổ và gọi hàm kiểm tra định nghĩa (`is_magic_brute` ở trên): `sorted(flat) == [1..9]`, tổng ba hàng, ba cột, hai đường chéo bằng nhau.

In [7]:
def count_brute(grid):
    return sum(1 for _, _, block in windows(grid) if is_magic_brute(block))


print(count_brute(ex1), count_brute(ex2))
assert count_brute(ex1) == 1 and count_brute(ex2) == 0

1 0


**Độ phức tạp:** `(row − 2)(col − 2)` cửa sổ × `O(1)` (kiểm tra 9 ô) = `O(row · col)`; `row, col ≤ 10` nên cùng lắm 64 cửa sổ. Bộ nhớ `O(1)`.

**Nút thắt:** không có nút thắt hiệu năng ở ràng buộc này; nhưng ta có thể **loại cửa sổ sai rất nhanh** nhờ tính chất toán học, và có thể tra bảng 8 ma phương.

## 6. Ý tưởng thuật toán / bước đột phá

**Chứng minh tâm bằng 5.** Gọi 9 ô là `a b c / d e f / g h i` (`e` là tâm). Tổng mỗi đường `= S`. Tổng mọi số `1..9` là `45 = 3S` nên `S = 15`. Cộng bốn đường qua tâm: hàng giữa `d + e + f`, cột giữa `b + e + h`, hai đường chéo `a + e + i` và `c + e + g`: tổng `4S = 60`. Mỗi ô khác tâm xuất hiện đúng một lần trong tổng đó (tám ô), còn tâm `e` xuất hiện **bốn lần**: `60 = (45 − e) + 4e = 45 + 3e`, suy ra `e = 5`. ∎

**Hệ quả:** mỗi cặp ô đối xứng qua tâm (như `a, i` hoặc `b, h`) có tổng `15 − 5 = 10`. Các cặp `(1,9), (2,8), (3,7), (4,6)` là bốn cặp có tổng `10`; bốn đường qua tâm dùng đúng các cặp này. Từ máy (liệt kê ở mục 3.5) ta quan sát thêm: bốn **góc** của mọi ma phương đều là số **chẵn** `2, 4, 6, 8` và bốn **cạnh** là số **lẻ** `1, 3, 7, 9`; ta dùng đúng quan sát này ở Cách 4 nhưng không cần chứng minh riêng vì máy đã liệt kê toàn bộ 8 ma phương.

Ba cách khai thác:

1. **Lọc nhanh bằng tâm:** nếu `block[1][1] != 5` loại ngay; nếu đúng, kiểm tra các đường còn lại.
2. **Tra bảng:** dựng sẵn 8 ma phương (một ma phương gốc cùng 8 phép đối xứng của hình vuông — 4 phép xoay × lật hoặc không) và kiểm tra cửa sổ có thuộc tập đó không.
3. **So khớp vòng ngoài:** đọc 8 ô viền theo chiều kim đồng hồ ra chuỗi; nếu tâm là `5` thì mọi ma phương có vòng ngoài là một **dịch vòng** của `4 9 2 7 6 1 8 3` hoặc của chuỗi đảo ngược của nó.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** lọc bằng tâm `5` + kiểm tra các đường, **Cách 3** tra cứu 8 ma phương dựng bằng đối xứng, **Cách 4** so khớp vòng ngoài.

### Cách 2 — Tâm bằng 5, rồi kiểm tra các đường còn lại

**Giải thích từng đoạn:**

1. Nếu `grid[i+1][j+1] != 5` → không phải ma phương (loại rất nhiều cửa sổ chỉ với một phép so sánh).
2. Chín số phải phân biệt và trong `1..9`: dùng `set`.
3. Các cặp đối xứng qua tâm có tổng `10`: `a + i = c + g = b + h = d + f = 10`.
4. Hai hàng ngoài cùng `a + b + c = 15` và `g + h + i = 15`, và hai cột ngoài `a + d + g = 15`, `c + f + i = 15`. (Hàng giữa, cột giữa, hai đường chéo đã đúng nhờ các cặp tổng 10 và tâm 5.)

In [8]:
def is_magic_center(block):
    (a, b, c), (d, e, f), (g, h, i) = block
    if e != 5:
        return False
    if set((a, b, c, d, e, f, g, h, i)) != set(range(1, 10)):
        return False
    if not (a + i == c + g == b + h == d + f == 10):                     # cặp đối xứng qua tâm
        return False
    return a + b + c == 15 and g + h + i == 15 and a + d + g == 15 and c + f + i == 15


def count_center(grid):
    return sum(1 for _, _, block in windows(grid) if is_magic_center(block))


print(count_center(ex1), count_center(ex2))
assert count_center(ex1) == 1 and count_center(ex2) == 0

1 0


In [9]:
# Code nộp LeetCode cho Cách 2
class SolutionCenter:
    def numMagicSquaresInside(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        ans = 0
        for i in range(m - 2):
            for j in range(n - 2):
                if grid[i + 1][j + 1] != 5:
                    continue
                a, b, c = grid[i][j:j + 3]
                d, e, f = grid[i + 1][j:j + 3]
                g, h, k = grid[i + 2][j:j + 3]
                if set((a, b, c, d, e, f, g, h, k)) != set(range(1, 10)):
                    continue
                if a + k == c + g == b + h == d + f == 10 and a + b + c == 15 and g + h + k == 15 and a + d + g == 15 and c + f + k == 15:
                    ans += 1
        return ans

**Độ phức tạp:** `O(row · col)`; hằng số nhỏ vì phần lớn cửa sổ bị loại bởi `tâm != 5`.

**Vì sao đủ:** nếu tâm `5`, chín số là hoán vị của `1..9` và các cặp đối xứng tổng `10`, thì hàng giữa `d + 5 + f = 15`, cột giữa `b + 5 + h = 15`, hai đường chéo `a + 5 + i = c + 5 + g = 15` tự đúng. Cộng thêm hai hàng ngoài và hai cột ngoài đúng `15` là đủ cả 8 đường.

### Cách 3 — Tra cứu 8 ma phương

**Ý tưởng:** chỉ có 8 ma phương `3 × 3`; dựng chúng một lần từ **ma phương gốc** (Lô Thư / Lo Shu) `[[2,7,6],[9,5,1],[4,3,8]]` bằng 4 phép xoay `90°` và phép lật ngang; mã hoá mỗi cửa sổ thành **tuple 9 số** rồi tra trong `set`. Mỗi cửa sổ tốn `O(1)` băm.

In [10]:
def rotate90(block):
    return tuple(zip(*block[::-1]))                      # xoay 90 độ theo chiều kim đồng hồ


def all_symmetries(block):
    out = set()
    cur = tuple(map(tuple, block))
    for _ in range(4):
        out.add(cur)
        out.add(tuple(row[::-1] for row in cur))         # lật ngang
        cur = rotate90(cur)
    return out


LO_SHU = [[2, 7, 6], [9, 5, 1], [4, 3, 8]]
MAGIC_SET = {tuple(v for row in sq for v in row) for sq in all_symmetries(LO_SHU)}
print("số ma phương dựng bằng đối xứng:", len(MAGIC_SET))
assert MAGIC_SET == {tuple(p) for p in magic_squares}          # trùng với kết quả vét cạn hoán vị
for sq in sorted(MAGIC_SET):
    print(sq[0:3], sq[3:6], sq[6:9])


def count_lookup(grid):
    m, n = len(grid), len(grid[0])
    ans = 0
    for i in range(m - 2):
        for j in range(n - 2):
            key = tuple(grid[i + k][j + t] for k in range(3) for t in range(3))
            ans += key in MAGIC_SET
    return ans


print(count_lookup(ex1), count_lookup(ex2))
assert count_lookup(ex1) == 1 and count_lookup(ex2) == 0

số ma phương dựng bằng đối xứng: 8
(2, 7, 6) (9, 5, 1) (4, 3, 8)
(2, 9, 4) (7, 5, 3) (6, 1, 8)
(4, 3, 8) (9, 5, 1) (2, 7, 6)
(4, 9, 2) (3, 5, 7) (8, 1, 6)
(6, 1, 8) (7, 5, 3) (2, 9, 4)
(6, 7, 2) (1, 5, 9) (8, 3, 4)
(8, 1, 6) (3, 5, 7) (4, 9, 2)
(8, 3, 4) (1, 5, 9) (6, 7, 2)
1 0


In [11]:
# Code nộp LeetCode cho Cách 3
MAGIC = {
    (2, 7, 6, 9, 5, 1, 4, 3, 8), (2, 9, 4, 7, 5, 3, 6, 1, 8), (4, 3, 8, 9, 5, 1, 2, 7, 6), (4, 9, 2, 3, 5, 7, 8, 1, 6),
    (6, 1, 8, 7, 5, 3, 2, 9, 4), (6, 7, 2, 1, 5, 9, 8, 3, 4), (8, 1, 6, 3, 5, 7, 4, 9, 2), (8, 3, 4, 1, 5, 9, 6, 7, 2),
}


class SolutionLookup:
    def numMagicSquaresInside(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        ans = 0
        for i in range(m - 2):
            for j in range(n - 2):
                key = tuple(grid[i + a][j + b] for a in range(3) for b in range(3))
                if key in MAGIC:
                    ans += 1
        return ans

In [12]:
assert MAGIC == MAGIC_SET, "bảng 8 ma phương viết tay phải trùng bảng dựng bằng đối xứng và bảng vét cạn"
print("bảng 8 ma phương viết tay trùng khớp:", MAGIC == MAGIC_SET == {tuple(p) for p in magic_squares})

bảng 8 ma phương viết tay trùng khớp: True


**Độ phức tạp:** `O(row · col)`; hằng số nhỏ (băm một tuple 9 số). Gọn nhất nhưng phụ thuộc vào việc **tin** bảng 8 phần tử — ta đã kiểm chứng bảng bằng vét cạn hoán vị ở mục 3.5.

### Cách 4 — So khớp vòng ngoài (ring) với `4 9 2 7 6 1 8 3`

**Ý tưởng khác:** đọc 8 ô viền theo chiều kim đồng hồ từ góc trên trái: `a b c f i h g d`. Với ma phương `4 9 2 / 3 5 7 / 8 1 6` vòng ngoài là `4 9 2 7 6 1 8 3`. Mọi ma phương `3 × 3` có tâm `5` và vòng ngoài là một **dịch vòng** của chuỗi `R = "49276183"` hoặc của chuỗi đảo `R[::-1]` (phép lật), **và** bốn góc (`a, c, i, g`, các vị trí `0, 2, 4, 6` của vòng) đều là số **chẵn** — điều kiện này loại các dịch vòng lẻ biến cạnh thành góc.

Cài đặt gọn: ma phương hợp lệ khi tâm `5`, các số ở góc chẵn, và chuỗi vòng ngoài xuất hiện trong `R + R` hoặc `R[::-1] + R[::-1]`.

In [13]:
RING = "49276183"


def ring_of(block):
    (a, b, c), (d, e, f), (g, h, i) = block
    return [a, b, c, f, i, h, g, d]                       # theo chiều kim đồng hồ từ góc trên trái


def is_magic_ring(block):
    if block[1][1] != 5:
        return False
    ring = ring_of(block)
    if any(v < 1 or v > 9 for v in ring) or any(v % 2 for v in ring[0::2]):      # bốn góc phải là số chẵn
        return False
    s = "".join(map(str, ring))
    return s in RING + RING or s in RING[::-1] + RING[::-1]


def count_ring(grid):
    return sum(1 for _, _, block in windows(grid) if is_magic_ring(block))


print(count_ring(ex1), count_ring(ex2))
assert count_ring(ex1) == 1 and count_ring(ex2) == 0
# kiểm chứng: is_magic_ring đúng đúng trên 8 ma phương trong mọi hoán vị 1..9? (làm đầy đủ ở mục 12D)
ring_ok = {tuple(p) for p in permutations(range(1, 10)) if is_magic_ring([p[0:3], p[3:6], p[6:9]])}
print("số hoán vị 1..9 mà is_magic_ring cho True:", len(ring_ok))
assert ring_ok == {tuple(p) for p in magic_squares}

1 0


số hoán vị 1..9 mà is_magic_ring cho True: 8


In [14]:
# Code nộp LeetCode cho Cách 4
class SolutionRing:
    def numMagicSquaresInside(self, grid: List[List[int]]) -> int:
        ring = "49276183"
        patterns = (ring + ring, ring[::-1] + ring[::-1])
        m, n = len(grid), len(grid[0])
        ans = 0
        for i in range(m - 2):
            for j in range(n - 2):
                if grid[i + 1][j + 1] != 5:
                    continue
                cells = [grid[i][j], grid[i][j + 1], grid[i][j + 2], grid[i + 1][j + 2],
                         grid[i + 2][j + 2], grid[i + 2][j + 1], grid[i + 2][j], grid[i + 1][j]]
                if any(v < 1 or v > 9 for v in cells) or any(v % 2 for v in cells[0::2]):
                    continue                                            # số ngoài 1..9 hoặc góc lẻ
                s = "".join(map(str, cells))
                if s in patterns[0] or s in patterns[1]:
                    ans += 1
        return ans

**Độ phức tạp:** `O(row · col)` với hằng số nhỏ (so khớp chuỗi 8 ký tự). Hay ở chỗ nó cho thấy **cấu trúc** của ma phương (vòng ngoài quay), nhưng khó nhớ và dễ sai (số `10..15` phải bị loại trước); tra bảng hay lọc tâm `5` thực dụng hơn.

## 8. TRACE từng bước

**Trace Cách 2** trên ví dụ 1 (`3 × 4`, có `(3 − 2)(4 − 2) = 2` cửa sổ):

| cửa sổ `(i, j)` | khối | tâm | chín số là `1..9`? | cặp đối xứng tổng `10`? | hàng/cột ngoài `= 15`? | kết luận |
|---|---|---|---|---|---|---|
| `(0, 0)` | `[[4,3,8],[9,5,1],[2,7,6]]` | `5` ✓ | có | `4+6 = 10`, `8+2 = 10`, `3+7 = 10`, `9+1 = 10` ✓ | `15, 15, 15, 15` ✓ | **ma phương** |
| `(0, 1)` | `[[3,8,4],[5,1,9],[7,6,2]]` | `1` ✗ | (không cần) | (không cần) | (không cần) | loại ngay vì tâm `≠ 5` |

Hàm trace in từng bước kiểm tra, kèm một số cửa sổ phản ví dụ khác:

In [15]:
def trace_center(grid):
    m, n = len(grid), len(grid[0])
    found = 0
    for i in range(m - 2):
        for j in range(n - 2):
            block = [grid[i + k][j:j + 3] for k in range(3)]
            print(f"cửa sổ {(i, j)}: {block}")
            (a, b, c), (d, e, f), (g, h, k_) = block
            if e != 5:
                print(f"   tâm = {e} != 5 -> loại")
                continue
            flat = [a, b, c, d, e, f, g, h, k_]
            if set(flat) != set(range(1, 10)):
                print(f"   chín số {sorted(flat)} không phải hoán vị 1..9 -> loại")
                continue
            pairs = (a + k_, c + g, b + h, d + f)
            if not all(p == 10 for p in pairs):
                print(f"   các cặp đối xứng {pairs} không đều 10 -> loại")
                continue
            lines = (a + b + c, g + h + k_, a + d + g, c + f + k_)
            if not all(l == 15 for l in lines):
                print(f"   hàng/cột ngoài {lines} không đều 15 -> loại")
                continue
            print(f"   cặp đối xứng {pairs}, hàng/cột ngoài {lines} -> MA PHƯƠNG")
            found += 1
    print("=> tổng số ma phương:", found)
    return found


assert trace_center(ex1) == 1
print()
semi_magic = [[9, 5, 1], [2, 7, 6], [4, 3, 8]]                     # đổi chỗ hai hàng của Lo Shu: hàng/cột đúng, chéo sai
all_fives = [[5, 5, 5], [5, 5, 5], [5, 5, 5]]
big_values = [[10, 15, 8], [11, 9, 7], [12, 3, 6]]
assert trace_center(semi_magic) == 0 and trace_center(all_fives) == 0

cửa sổ (0, 0): [[4, 3, 8], [9, 5, 1], [2, 7, 6]]
   cặp đối xứng (10, 10, 10, 10), hàng/cột ngoài (15, 15, 15, 15) -> MA PHƯƠNG
cửa sổ (0, 1): [[3, 8, 4], [5, 1, 9], [7, 6, 2]]
   tâm = 1 != 5 -> loại
=> tổng số ma phương: 1

cửa sổ (0, 0): [[9, 5, 1], [2, 7, 6], [4, 3, 8]]
   tâm = 7 != 5 -> loại
=> tổng số ma phương: 0
cửa sổ (0, 0): [[5, 5, 5], [5, 5, 5], [5, 5, 5]]
   chín số [5, 5, 5, 5, 5, 5, 5, 5, 5] không phải hoán vị 1..9 -> loại
=> tổng số ma phương: 0


**Trace Cách 3 (tra bảng)** cho ví dụ 1: mã hoá từng cửa sổ thành tuple 9 số và tra `MAGIC`: cửa sổ `(0,0)` → `(4,3,8,9,5,1,2,7,6)` ∈ `MAGIC` ✓; cửa sổ `(0,1)` → `(3,8,4,5,1,9,7,6,2)` ∉ `MAGIC` ✗. Hàm in từng bước:

In [16]:
def trace_lookup(grid):
    m, n = len(grid), len(grid[0])
    ans = 0
    for i in range(m - 2):
        for j in range(n - 2):
            key = tuple(grid[i + a][j + b] for a in range(3) for b in range(3))
            hit = key in MAGIC
            ans += hit
            print(f"cửa sổ {(i, j)}: key = {key} -> {'có trong bảng 8 ma phương' if hit else 'không có'}")
    print("=> đáp án", ans)
    return ans


assert trace_lookup(ex1) == 1

cửa sổ (0, 0): key = (4, 3, 8, 9, 5, 1, 2, 7, 6) -> có trong bảng 8 ma phương
cửa sổ (0, 1): key = (3, 8, 4, 5, 1, 9, 7, 6, 2) -> không có
=> đáp án 1


## 9. Code nộp LeetCode

Chọn **tâm bằng 5 + kiểm tra** (đủ nhanh và **tự chứng minh được**, không cần tin vào bảng). Giữ nguyên `class Solution`, `numMagicSquaresInside(self, grid)`:

In [17]:
from typing import List


class Solution:
    def numMagicSquaresInside(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        ans = 0
        for i in range(m - 2):
            for j in range(n - 2):
                if grid[i + 1][j + 1] != 5:                     # ma phương 3x3 luôn có tâm 5
                    continue
                a, b, c = grid[i][j:j + 3]
                d, e, f = grid[i + 1][j:j + 3]
                g, h, k = grid[i + 2][j:j + 3]
                if set((a, b, c, d, e, f, g, h, k)) != set(range(1, 10)):   # phân biệt, đủ 1..9
                    continue
                # các cặp đối xứng qua tâm tổng 10 và hai hàng/cột ngoài tổng 15
                if a + k == c + g == b + h == d + f == 10 and a + b + c == 15 and g + h + k == 15 \
                        and a + d + g == 15 and c + f + k == 15:
                    ans += 1
        return ans


print(Solution().numMagicSquaresInside(ex1), Solution().numMagicSquaresInside(ex2))
assert Solution().numMagicSquaresInside(ex1) == 1 and Solution().numMagicSquaresInside(ex2) == 0

1 0


## 10. Vì sao đúng

**(a) Định nghĩa ⇒ tâm `5`.** Đã chứng minh ở mục 6: nếu mọi đường tổng `S` và chín số là `1..9`, thì `3S = 45` nên `S = 15` và `4S = 60 = 45 + 3e` cho `e = 5`.

**(b) Điều kiện của Cách 2 đủ.** Giả sử tâm `e = 5`, chín số là hoán vị `1..9`, các cặp đối xứng `a + i = c + g = b + h = d + f = 10` và `a + b + c = g + h + i = a + d + g = c + f + i = 15`. Khi đó:

- hàng giữa `d + e + f = 10 + 5 = 15`; cột giữa `b + e + h = 15`;
- hai đường chéo `a + e + i = 10 + 5 = 15`, `c + e + g = 15`;
- hai hàng ngoài và hai cột ngoài đều `15` theo giả thiết.

Vậy đủ 8 đường tổng `15`. Điều ngược lại (ma phương ⇒ các điều kiện đó) hiển nhiên từ định nghĩa và (a). ∎

**(c) Chỉ có 8 ma phương.** Máy duyệt 362 880 hoán vị tìm đúng 8 (mục 3.5); chúng là 4 phép xoay và 4 phép xoay-của-ảnh-lật của Lo Shu (8 phép đối xứng của hình vuông áp lên một ma phương gốc cho 8 kết quả khác nhau — đã kiểm chứng `all_symmetries` cho đúng tập 8). Do đó Cách 3 đúng khi cửa sổ có chín số trùng một trong 8 bộ này, mà bất kỳ cửa sổ nào có số ngoài `1..9` hoặc lặp thì không thể trùng.

**(d) Cách 4.** Cấu trúc vòng ngoài (tâm `5`, bốn góc chẵn, viền là dịch vòng của `49276183` hoặc chuỗi đảo): kiểm tra bằng máy ở mục 7 — trong mọi hoán vị `1..9`, `is_magic_ring` cho đúng đúng 8 ma phương.

Ta kiểm chứng toàn bộ ở mục 12: cả bốn cách khớp oracle trên mọi hoán vị, trên các biến thể "đổi một ô" của mọi ma phương và trên hàng nghìn lưới ngẫu nhiên.

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Kiểm tra định nghĩa mỗi cửa sổ | `O(row·col)` | `O(1)` | rõ ràng, hằng số lớn nhất |
| 2. Tâm `5` + kiểm tra | `O(row·col)` | `O(1)` | **đáp án chính**; nhanh vì loại sớm |
| 3. Tra bảng 8 ma phương | `O(row·col)` | `O(1)` (8 bộ) | gọn nhất; cần tin bảng (đã chứng minh bằng máy) |
| 4. So khớp vòng ngoài | `O(row·col)` | `O(1)` | cho thấy cấu trúc; dễ sai |

Với `row, col ≤ 10` chỉ có tối đa `64` cửa sổ nên mọi cách đều tức thời. Ta vẫn đo để so sánh hằng số (lưới lớn hơn giới hạn đề):

In [18]:
def random_grid(rng, m, n, hi=15):
    return [[rng.randint(0, hi) for _ in range(n)] for _ in range(m)]


def plant(grid, i, j, square):
    for a in range(3):
        for b in range(3):
            grid[i + a][j + b] = square[3 * a + b]


def bench(fn, grid, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(grid)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(840)
g = random_grid(rng, 300, 300, 9)                          # lưới lớn hơn đề, nhiều số 1..9
sq = sorted(MAGIC_SET)
for k in range(0, 297, 3):
    plant(g, k, (k * 7) % 294, sq[k % 8])
ref = count_brute(g)
print("lưới 300x300 (ngoài ràng buộc, chỉ để so hằng số): đáp án", ref)
for name, fn in [("brute", count_brute), ("center", count_center), ("lookup", count_lookup), ("ring", count_ring)]:
    assert fn(g) == ref
    print(f"  {name:7s} {bench(fn, g) * 1000:8.1f} ms")

lưới 300x300 (ngoài ràng buộc, chỉ để so hằng số): đáp án 99


  brute      104.2 ms
  center      40.3 ms


  lookup      85.3 ms
  ring        39.8 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ: cài đặt sai phải bị bắt, (D) **vét cạn**: mọi hoán vị của `1..9` và mọi biến thể "đổi một ô" của 8 ma phương, (E) ngẫu nhiên có seed + ma phương được gieo vào, (F) giới hạn lớn nhất. **Oracle:** `count_brute` (kiểm tra định nghĩa) và tập ma phương do vét cạn hoán vị.

### A. Ví dụ của đề

In [19]:
IMPLS = {
    "brute": count_brute,
    "center": count_center,
    "lookup": count_lookup,
    "ring": count_ring,
    "Solution": lambda g: Solution().numMagicSquaresInside(g),
    "SolutionCenter": lambda g: SolutionCenter().numMagicSquaresInside(g),
    "SolutionLookup": lambda g: SolutionLookup().numMagicSquaresInside(g),
    "SolutionRing": lambda g: SolutionRing().numMagicSquaresInside(g),
}


def check(grid, want=None, names=None):
    if want is None:
        want = count_brute(grid)
    for name in (names or IMPLS):
        got = IMPLS[name]([row[:] for row in grid])
        assert got == want, (name, grid, got, want)


check(ex1, 1)
check(ex2, 0)
print("ví dụ 1, 2: cả", len(IMPLS), "cài đặt đúng")

ví dụ 1, 2: cả 8 cài đặt đúng


### B. Edge case

- Lưới nhỏ hơn `3` ở một chiều (`1 × 1`, `2 × 5`, `5 × 2`) → `0`.
- Đúng một cửa sổ `3 × 3`: là ma phương (cả 8 loại) và không phải.
- Lưới `3 × 3` toàn `5` (cùng tổng, trùng số).
- Lưới `3 × 4` có ma phương ở cột đầu (cửa sổ thứ hai chồng lên nhưng không hợp lệ).
- Ma phương cạnh nhau (`3 × 6`: hai ma phương liền).
- Số ngoài `1..9` (`0`, `10..15`) làm cửa sổ không hợp lệ dù tổng bằng nhau.
- Tổng bằng nhau nhưng số không phân biệt.

In [20]:
sq8 = sorted(MAGIC_SET)
as_grid = lambda sq: [list(sq[0:3]), list(sq[3:6]), list(sq[6:9])]
check([[8]], 0)
check([[0] * 5] * 2, 0)
check([[5] * 2 for _ in range(5)], 0)
for sq in sq8:
    check(as_grid(sq), 1)                                       # mỗi trong 8 ma phương đứng một mình
check([[5, 5, 5], [5, 5, 5], [5, 5, 5]], 0)                      # toàn 5: tổng bằng nhau nhưng trùng số
check([[9, 5, 1], [2, 7, 6], [4, 3, 8]], 0)                      # ma phương bán phần
check([[x + 1 for x in row] for row in as_grid(sq8[0])], 0)      # cộng 1 mọi ô: tổng bằng nhau nhưng vượt khoảng 1..9
two = [as_grid(sq8[0])[r] + as_grid(sq8[3])[r] for r in range(3)]
check(two, 2)                                                    # hai ma phương cạnh nhau (3x6): hai cửa sổ đầu cuối
print("3x6 có hai ma phương cạnh nhau:", count_brute(two))
overlap = [[2, 7, 6, 2], [9, 5, 1, 9], [4, 3, 8, 4]]
check(overlap, 1)
big_vals = [[15, 15, 15], [15, 15, 15], [15, 15, 15]]
check(big_vals, 0)
print("edge case: tất cả cài đặt đúng")

3x6 có hai ma phương cạnh nhau: 2
edge case: tất cả cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [21]:
def bug_no_diagonals(grid):                       # chỉ kiểm hàng và cột
    m, n = len(grid), len(grid[0])
    ans = 0
    for i in range(m - 2):
        for j in range(n - 2):
            block = [grid[i + k][j:j + 3] for k in range(3)]
            flat = [v for r in block for v in r]
            if sorted(flat) == list(range(1, 10)) and all(sum(r) == 15 for r in block) and all(sum(c) == 15 for c in zip(*block)):
                ans += 1
    return ans


def bug_no_distinct(grid):                        # chỉ kiểm các tổng, không kiểm số phân biệt / trong 1..9
    m, n = len(grid), len(grid[0])
    ans = 0
    for i in range(m - 2):
        for j in range(n - 2):
            b = [grid[i + k][j:j + 3] for k in range(3)]
            s = sum(b[0])
            if (all(sum(r) == s for r in b) and all(sum(c) == s for c in zip(*b))
                    and b[0][0] + b[1][1] + b[2][2] == s and b[0][2] + b[1][1] + b[2][0] == s):
                ans += 1
    return ans


def bug_one_diagonal(grid):                       # chỉ kiểm đường chéo chính
    m, n = len(grid), len(grid[0])
    ans = 0
    for i in range(m - 2):
        for j in range(n - 2):
            b = [grid[i + k][j:j + 3] for k in range(3)]
            if (sorted(v for r in b for v in r) == list(range(1, 10)) and all(sum(r) == 15 for r in b)
                    and all(sum(c) == 15 for c in zip(*b)) and b[0][0] + b[1][1] + b[2][2] == 15):
                ans += 1
    return ans


def bug_center_only(grid):                        # tin rằng 'tâm = 5 và phân biệt 1..9' là đủ
    m, n = len(grid), len(grid[0])
    ans = 0
    for i in range(m - 2):
        for j in range(n - 2):
            b = [grid[i + k][j:j + 3] for k in range(3)]
            if b[1][1] == 5 and sorted(v for r in b for v in r) == list(range(1, 10)):
                ans += 1
    return ans


def bug_range_off_by_one(grid):                   # range(m - 3): bỏ sót hàng/cột cuối
    m, n = len(grid), len(grid[0])
    ans = 0
    for i in range(m - 3):
        for j in range(n - 3):
            if is_magic_brute([grid[i + k][j:j + 3] for k in range(3)]):
                ans += 1
    return ans


BUGS = [bug_no_diagonals, bug_no_distinct, bug_one_diagonal, bug_center_only, bug_range_off_by_one]
rng = random.Random(1)
tests = []
for _ in range(1500):
    g = random_grid(rng, rng.randint(3, 6), rng.randint(3, 6), 9)
    if rng.random() < 0.7:
        i, j = rng.randint(0, len(g) - 3), rng.randint(0, len(g[0]) - 3)
        plant(g, i, j, rng.choice(sq8))
        if rng.random() < 0.5:                         # làm hỏng một ô để tạo ca "suýt" ma phương
            g[i + rng.randint(0, 2)][j + rng.randint(0, 2)] = rng.randint(0, 15)
    tests.append(g)
# lưới 1..9 có hàng, cột và đường chéo CHÍNH đúng 15 nhưng đường chéo phụ sai (tìm bằng vét cạn hoán vị)
one_diag = [as_grid(p) for p in permutations(range(1, 10))
            if all(sum(p[3 * r:3 * r + 3]) == 15 for r in range(3)) and all(sum(p[c::3]) == 15 for c in range(3))
            and p[0] + p[4] + p[8] == 15 and p[2] + p[4] + p[6] != 15]
print("số lưới hoán vị 1..9 có hàng, cột, chéo chính = 15 nhưng chéo phụ sai:", len(one_diag))
extra = [[[9, 5, 1], [2, 7, 6], [4, 3, 8]], [[5, 5, 5]] * 3, [[3, 8, 4], [5, 1, 9], [7, 6, 2]]] + one_diag
tests += extra
for bug in BUGS:
    bad = next((g for g in tests if bug(g) != count_brute(g)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:22s} sai trên lưới {len(bad)}x{len(bad[0])}: ra {bug(bad)}, đúng {count_brute(bad)}")

số lưới hoán vị 1..9 có hàng, cột, chéo chính = 15 nhưng chéo phụ sai: 16


bug_no_diagonals       sai trên lưới 3x3: ra 1, đúng 0
bug_no_distinct        sai trên lưới 3x3: ra 1, đúng 0
bug_one_diagonal       sai trên lưới 3x3: ra 1, đúng 0
bug_center_only        sai trên lưới 6x6: ra 2, đúng 1
bug_range_off_by_one   sai trên lưới 4x3: ra 0, đúng 1


### D. Vét cạn

**D1.** Mọi hoán vị của `1..9` (362 880) làm lưới `3 × 3`: mọi cài đặt phải cho `1` đúng với 8 ma phương và `0` với phần còn lại.

**D2.** Với mỗi ma phương trong 8 cái, thay **một ô** bằng mọi giá trị `0..15` (khác giá trị cũ): `8 × 9 × 15 = 1080` lưới "suýt ma phương" — tất cả phải cho `0`.

**D3.** Mọi lưới `3 × 3` với giá trị trong `{1, 5, 9}` (`3^9 = 19683`) — nhiều tổng bằng nhau, nhiều số trùng.

In [22]:
t0 = time.perf_counter()
magic_set_local = {tuple(p) for p in magic_squares}
fast = ["center", "lookup", "ring"]
count_by = {name: 0 for name in fast + ["brute"]}
for p in permutations(range(1, 10)):
    block = [list(p[0:3]), list(p[3:6]), list(p[6:9])]
    want = p in magic_set_local
    # chỉ gọi các hàm 'is_magic_*' (đơn vị) để nhanh; các lớp nộp bài kiểm ở các nhóm sau
    assert is_magic_brute(block) == want
    assert is_magic_center(block) == want
    assert is_magic_ring(block) == want
    assert (p in MAGIC_SET) == want
    count_by["brute"] += want
print("D1: mọi", 362880, "hoán vị 1..9: 4 phương pháp kiểm tra khớp tập ma phương; số ma phương =", count_by["brute"], "; mất", round(time.perf_counter() - t0, 1), "s")
assert count_by["brute"] == 8

D1: mọi 362880 hoán vị 1..9: 4 phương pháp kiểm tra khớp tập ma phương; số ma phương = 8 ; mất 1.1 s


In [23]:
t0 = time.perf_counter()
cnt = 0
for sq in sq8:
    for pos in range(9):
        for v in range(16):
            if v == sq[pos]:
                continue
            mod = list(sq)
            mod[pos] = v
            g = as_grid(mod)
            for name in IMPLS:
                assert IMPLS[name](g) == 0, (name, mod)
            cnt += 1
print("D2:", cnt, "lưới 'đổi một ô' của 8 ma phương: mọi cài đặt cho 0")

cnt = 0
for t in itertools.product((1, 5, 9), repeat=9):
    g = [list(t[0:3]), list(t[3:6]), list(t[6:9])]
    want = 1 if t in MAGIC_SET else 0
    assert want == 0                                      # chỉ có 3 giá trị khác nhau: không thể là hoán vị 1..9
    for name in ("brute", "center", "lookup", "ring", "Solution", "SolutionLookup"):
        assert IMPLS[name](g) == want, (name, t)
    cnt += 1
print("D3:", cnt, "lưới 3x3 giá trị {1,5,9}: mọi cài đặt cho 0;", round(time.perf_counter() - t0, 1), "s")

D2: 1080 lưới 'đổi một ô' của 8 ma phương: mọi cài đặt cho 0
D3: 19683 lưới 3x3 giá trị {1,5,9}: mọi cài đặt cho 0; 0.2 s


### E. Ngẫu nhiên có seed, đúng ràng buộc

Lưới ngẫu nhiên kích thước `1..10`, giá trị `0..15`, nhiều lần **gieo** một hoặc vài ma phương (có thể chồng lên nhau, ghi đè), kèm các biến thể "suýt".

In [24]:
rng = random.Random(20240840)
count = planted = 0
for _ in range(3000):
    m, n = rng.randint(1, 10), rng.randint(1, 10)
    g = random_grid(rng, m, n, rng.choice([9, 15]))
    if m >= 3 and n >= 3:
        for _ in range(rng.randint(0, 4)):
            i, j = rng.randint(0, m - 3), rng.randint(0, n - 3)
            plant(g, i, j, rng.choice(sq8))
            planted += 1
        if rng.random() < 0.3:
            g[rng.randrange(m)][rng.randrange(n)] = rng.randint(0, 15)
    want = count_brute(g)
    for name in IMPLS:
        assert IMPLS[name](g) == want, (name, g)
    count += 1
print(count, "lưới ngẫu nhiên (đã gieo", planted, "ma phương): mọi cài đặt trùng oracle")

# oracle độc lập: số cửa sổ có tuple 9 số nằm trong tập vét cạn hoán vị
for _ in range(500):
    g = random_grid(rng, rng.randint(3, 8), rng.randint(3, 8), 9)
    for _ in range(rng.randint(0, 3)):
        plant(g, rng.randint(0, len(g) - 3), rng.randint(0, len(g[0]) - 3), rng.choice(sq8))
    m, n = len(g), len(g[0])
    want = sum(tuple(g[i + a][j + b] for a in range(3) for b in range(3)) in magic_set_local for i in range(m - 2) for j in range(n - 2))
    assert Solution().numMagicSquaresInside(g) == want
print("đối chiếu độc lập với tập ma phương vét cạn hoán vị: OK trên 500 lưới")

3000 lưới ngẫu nhiên (đã gieo 3872 ma phương): mọi cài đặt trùng oracle
đối chiếu độc lập với tập ma phương vét cạn hoán vị: OK trên 500 lưới


### F. Giới hạn lớn nhất

`10 × 10`: `64` cửa sổ. Các ca: toàn một số, đầy ma phương lát gạch (các ma phương sát nhau tạo các cửa sổ phụ không hợp lệ), mọi ô `5`, ma phương ở bốn góc.

In [25]:
m = n = 10
tiled = [[0] * n for _ in range(m)]
for i in range(0, 9, 3):
    for j in range(0, 9, 3):
        plant(tiled, i, j, sq8[(i + j) % 8])
corners = [[0] * n for _ in range(m)]
for (i, j), sq in zip([(0, 0), (0, 7), (7, 0), (7, 7)], sq8):
    plant(corners, i, j, sq)
cases = {
    "toàn 5": [[5] * n for _ in range(m)],
    "toàn 15": [[15] * n for _ in range(m)],
    "toàn 0": [[0] * n for _ in range(m)],
    "lát 3x3 (9 ma phương)": tiled,
    "ma phương ở 4 góc": corners,
}
for name, g in cases.items():
    want = count_brute(g)
    check(g, want)
    print(f"{name:24s}: đáp án {want}")
assert count_brute(cases["lát 3x3 (9 ma phương)"]) >= 9
assert count_brute(cases["ma phương ở 4 góc"]) == 4

toàn 5                  : đáp án 0
toàn 15                 : đáp án 0
toàn 0                  : đáp án 0
lát 3x3 (9 ma phương)   : đáp án 9
ma phương ở 4 góc       : đáp án 4


## 13. Các lỗi người mới hay gặp

### Lỗi 1: bỏ sót đường chéo (ma phương bán phần)

Chỉ kiểm hàng và cột cho ra các **ma phương bán phần** (semi-magic): ví dụ đổi chỗ hai hàng của ma phương gốc giữ nguyên tổng hàng và cột nhưng phá đường chéo.

In [26]:
semi = [[9, 5, 1], [2, 7, 6], [4, 3, 8]]
print("hàng:", [sum(r) for r in semi], "| cột:", [sum(c) for c in zip(*semi)], "| chéo:", semi[0][0] + semi[1][1] + semi[2][2], semi[0][2] + semi[1][1] + semi[2][0])
print("bug_no_diagonals:", bug_no_diagonals(semi), "| đúng:", Solution().numMagicSquaresInside(semi))
assert bug_no_diagonals(semi) == 1 and Solution().numMagicSquaresInside(semi) == 0

hàng: [15, 15, 15] | cột: [15, 15, 15] | chéo: 24 12
bug_no_diagonals: 1 | đúng: 0


**Vì sao sai:** đường chéo cũng là điều kiện của định nghĩa. **Đúng:** kiểm cả `8` đường (3 hàng, 3 cột, 2 chéo).

### Lỗi 2: quên điều kiện phân biệt và `1..9`

Lưới toàn `5` có mọi tổng bằng `15`; lưới "ma phương cộng 1" có mọi tổng bằng `18` nhưng dùng `2..10`. Cả hai đều **không** phải ma phương theo đề.

In [27]:
fives = [[5, 5, 5], [5, 5, 5], [5, 5, 5]]
shifted = [[v + 1 for v in row] for row in as_grid(sq8[0])]
print("toàn 5      :", bug_no_distinct(fives), "| đúng:", Solution().numMagicSquaresInside(fives))
print("ma phương +1:", bug_no_distinct(shifted), "| đúng:", Solution().numMagicSquaresInside(shifted), shifted)
assert bug_no_distinct(fives) == 1 and bug_no_distinct(shifted) == 1
assert Solution().numMagicSquaresInside(fives) == 0 and Solution().numMagicSquaresInside(shifted) == 0

toàn 5      : 1 | đúng: 0
ma phương +1: 1 | đúng: 0 [[3, 8, 7], [10, 6, 2], [5, 4, 9]]


**Vì sao sai:** định nghĩa đòi **chín số phân biệt từ 1 đến 9**. **Đúng:** `sorted(flat) == list(range(1, 10))` hoặc `set(flat) == set(range(1, 10))`.

### Lỗi 3: chỉ kiểm một đường chéo

Kiểm đường chéo chính nhưng quên đường chéo phụ (hoặc ngược lại): vẫn lọt một số lưới sai.

In [28]:
witness = one_diag[0]
print(witness)
print("hàng:", [sum(r) for r in witness], "| cột:", [sum(c) for c in zip(*witness)],
      "| chéo chính:", witness[0][0] + witness[1][1] + witness[2][2], "| chéo phụ:", witness[0][2] + witness[1][1] + witness[2][0])
print("bug_one_diagonal:", bug_one_diagonal(witness), "| đúng:", Solution().numMagicSquaresInside(witness))
assert bug_one_diagonal(witness) == 1 and Solution().numMagicSquaresInside(witness) == 0

[[2, 4, 9], [6, 8, 1], [7, 3, 5]]
hàng: [15, 15, 15] | cột: [15, 15, 15] | chéo chính: 15 | chéo phụ: 24
bug_one_diagonal: 1 | đúng: 0


**Vì sao sai:** cả hai đường chéo đều phải bằng tổng. **Đúng:** kiểm đủ hai đường.

### Lỗi 4: tin rằng "tâm = 5" đủ

`tâm = 5` chỉ là **điều kiện cần**. Có nhiều lưới hoán vị của `1..9` với tâm `5` mà không phải ma phương (đổi hai ô góc, chẳng hạn).

In [29]:
count_center5 = sum(1 for p in permutations(range(1, 10)) if p[4] == 5)
print("số hoán vị 1..9 có tâm 5:", count_center5, "| trong đó là ma phương:", sum(1 for p in permutations(range(1, 10)) if p[4] == 5 and p in MAGIC_SET))
g = as_grid((4, 3, 8, 9, 5, 1, 6, 7, 2))      # đổi hai góc dưới của Lo Shu
print(g, "-> bug_center_only:", bug_center_only(g), "| đúng:", Solution().numMagicSquaresInside(g))
assert bug_center_only(g) == 1 and Solution().numMagicSquaresInside(g) == 0

số hoán vị 1..9 có tâm 5: 40320 | trong đó là ma phương: 8
[[4, 3, 8], [9, 5, 1], [6, 7, 2]] -> bug_center_only: 1 | đúng: 0


**Vì sao sai:** tâm `5` + hoán vị `1..9` có tới `8! = 40 320` lưới, chỉ `8` trong đó là ma phương. **Đúng:** thêm các cặp đối xứng tổng `10` và các hàng/cột ngoài tổng `15`.

### Lỗi 5: lệch chỉ số khi trượt cửa sổ

Cửa sổ `3 × 3` cần `i + 2 < m`, tức `i ≤ m − 3`, nên `range(m − 2)`. Viết `range(m − 3)` bỏ sót hàng cuối cùng của các cửa sổ (và cột cuối).

In [30]:
g = as_grid(sq8[2])                              # đúng một cửa sổ 3x3 duy nhất
print("range(m-3) trên lưới 3x3:", bug_range_off_by_one(g), "| đúng:", Solution().numMagicSquaresInside(g))
g2 = [[0, 0, 0, 0]] + as_grid(sq8[1])             # ma phương ở hàng cuối
print("ma phương ở hàng cuối :", bug_range_off_by_one(g2), "| đúng:", Solution().numMagicSquaresInside(g2))
assert bug_range_off_by_one(g) == 0 and Solution().numMagicSquaresInside(g) == 1

range(m-3) trên lưới 3x3: 0 | đúng: 1
ma phương ở hàng cuối : 0 | đúng: 1


**Vì sao sai:** `range(m − 3)` dừng ở `i = m − 4`, không bao giờ xét cửa sổ cuối `i = m − 3`. **Đúng:** `range(m − 2)` (với `m < 3` là `range` rỗng, tự động `0`).

### Lỗi 6 (bonus): gán `a, b, c = grid[i][j:j+3]` khi hàng ngắn hơn

Nếu vô tình dùng cửa sổ vượt biên, lát cắt trả về **ít hơn 3 phần tử** và phép gán nhiều biến báo lỗi `ValueError`. Vòng lặp `range(m − 2)`, `range(n − 2)` đảm bảo không xảy ra.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "đếm các khối con có cấu trúc đặc biệt trong lưới nhỏ" → **duyệt mọi cửa sổ** và kiểm tra điều kiện; nếu cấu trúc **hiếm** (như ma phương), có thể **tiền xử lý tập các mẫu hợp lệ** rồi tra cứu.

**Mẹo:**

1. Dùng **bất biến toán học** để loại nhanh (tâm `5`, tổng `15`, cặp đối xứng tổng `10`).
2. Nếu số mẫu hợp lệ **rất ít**, vét cạn để **chứng minh** con số đó (ở đây `9! → 8`), rồi tra bảng.
3. Một ma phương sinh ra các ma phương khác bằng **đối xứng hình học** (xoay `90°`, lật): 8 phép biến đổi của hình vuông.
4. Kiểm tra "phân biệt và trong `1..9`" gọn bằng `sorted(flat) == list(range(1, 10))`.
5. Cẩn thận hai đường chéo; và đừng quên rằng điều kiện cần (tâm `5`) khác điều kiện đủ.
6. Trượt cửa sổ: `range(m − k + 1)`.

**Họ bài cùng pattern:** Valid Sudoku (36), Valid Square (593), Number of Submatrices That Sum to Target (1074), Largest 1-Bordered Square (1139), Spiral Matrix (54).

## 15. Biến thể

1. **Ma phương `n × n`** bất kỳ (cửa sổ `n × n`): chín số → `n²` số `1..n²`, tổng mỗi đường `n(n² + 1)/2`.
2. **Đếm ma phương bán phần** (chỉ hàng và cột): đáp án nhiều hơn.
3. **Cho phép các số lặp** hoặc **dải số khác** `[lo, lo + 8]`: dịch hằng số.
4. **Trả về vị trí** các ma phương (danh sách `(i, j)`).
5. **Sinh** mọi ma phương `3 × 3` (đúng 8) bằng đối xứng.

Cài đặt biến thể 1 (kiểm tra `n × n`) và 4, đối chiếu với bài gốc (`n = 3`) và ma phương Dürer `4 × 4`:

In [31]:
def is_magic_n(block):
    n = len(block)
    target = n * (n * n + 1) // 2
    flat = [v for r in block for v in r]
    if sorted(flat) != list(range(1, n * n + 1)):
        return False
    return (all(sum(r) == target for r in block)
            and all(sum(c) == target for c in zip(*block))
            and sum(block[i][i] for i in range(n)) == target
            and sum(block[i][n - 1 - i] for i in range(n)) == target)


def count_magic_n(grid, n):
    m, w = len(grid), len(grid[0])
    return sum(is_magic_n([grid[i + a][j:j + n] for a in range(n)]) for i in range(m - n + 1) for j in range(w - n + 1))


durer = [[16, 3, 2, 13], [5, 10, 11, 8], [9, 6, 7, 12], [4, 15, 14, 1]]
print("Dürer 4x4 là ma phương bậc 4:", is_magic_n(durer), "| tổng mỗi đường:", sum(durer[0]))
assert is_magic_n(durer) and is_magic_n(as_grid(sq8[0])) and not is_magic_n([[9, 5, 1], [2, 7, 6], [4, 3, 8]])
rng = random.Random(15)
for _ in range(300):
    g = random_grid(rng, rng.randint(3, 8), rng.randint(3, 8), 9)
    plant(g, rng.randint(0, len(g) - 3), rng.randint(0, len(g[0]) - 3), rng.choice(sq8)) if len(g) >= 3 and len(g[0]) >= 3 else None
    assert count_magic_n(g, 3) == Solution().numMagicSquaresInside(g)
print("count_magic_n(grid, 3) khớp bài gốc trên 300 lưới")
big = [[0] * 6 for _ in range(6)]
for a in range(4):
    for b in range(4):
        big[1 + a][1 + b] = durer[a][b]
print("lưới 6x6 chứa Dürer ở giữa: số ma phương 4x4 =", count_magic_n(big, 4))
assert count_magic_n(big, 4) == 1


def magic_positions(grid):
    """Biến thể 4: danh sách góc trên trái (i, j) của các ma phương 3x3."""
    m, n = len(grid), len(grid[0])
    return [(i, j) for i in range(m - 2) for j in range(n - 2) if is_magic_brute([grid[i + a][j:j + 3] for a in range(3)])]


print("vị trí trong ví dụ 1:", magic_positions(ex1))
assert magic_positions(ex1) == [(0, 0)] and len(magic_positions(two)) == 2

Dürer 4x4 là ma phương bậc 4: True | tổng mỗi đường: 34
count_magic_n(grid, 3) khớp bài gốc trên 300 lưới
lưới 6x6 chứa Dürer ở giữa: số ma phương 4x4 = 1
vị trí trong ví dụ 1: [(0, 0)]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Kiểm tra định nghĩa | `set`/`sorted` + 8 tổng | `O(row·col)` | `O(1)` | Dễ đúng; làm oracle |
| 2. Tâm `5` + kiểm tra | Bất biến toán học loại sớm | `O(row·col)` | `O(1)` | **Đáp án chính**; có chứng minh |
| 3. Tra bảng 8 ma phương | `tuple` 9 số ∈ tập 8 phần tử | `O(row·col)` | `O(1)` | Ngắn nhất; cần tin bảng (đã chứng minh bằng vét cạn hoán vị) |
| 4. So khớp vòng ngoài | Viền ∈ dịch vòng `49276183` | `O(row·col)` | `O(1)` | Cho thấy cấu trúc; dễ sai |

Cả bốn khớp nhau và khớp oracle trên mọi `9!` hoán vị, 1080 biến thể "đổi một ô", 19 683 lưới `{1,5,9}` và hàng nghìn lưới ngẫu nhiên (mục 12).

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi mới chạy:

In [32]:
b = [[8, 1, 6], [3, 5, 7], [4, 9, 2]]
print([sum(r) for r in b], [sum(c) for c in zip(*b)], b[0][0] + b[1][1] + b[2][2], b[0][2] + b[1][1] + b[2][0])
print(Solution().numMagicSquaresInside(b), Solution().numMagicSquaresInside([[8, 1, 6], [3, 5, 7], [4, 2, 9]]))
print(len(list(windows([[0] * 7] * 5))))

[15, 15, 15] [15, 15, 15] 15 15
1 0
15


### Bài 2 (tính tay)

- (a) Kiểm tra `[[6, 1, 8], [7, 5, 3], [2, 9, 4]]`: các tổng hàng, cột, chéo và có phải ma phương không?
- (b) Lưới `4 × 5` có bao nhiêu cửa sổ `3 × 3`?
- (c) Vì sao `[[2, 7, 6], [9, 5, 1], [4, 3, 9]]` (đổi `8` thành `9`) không phải ma phương? Chỉ ra **hai** lý do.

Tự giải rồi kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện hàm kiểm tra một khối `3 × 3`:

In [33]:
def is_magic_todo(block):
    flat = [v for r in block for v in r]
    # TODO 1: nếu flat không phải hoán vị của 1..9 thì trả False
    # TODO 2: kiểm tra 3 hàng, 3 cột và 2 đường chéo đều có tổng 15
    return None


# Khi làm xong, bỏ comment để kiểm tra:
# assert is_magic_todo([[4, 3, 8], [9, 5, 1], [2, 7, 6]]) is True
# assert is_magic_todo([[9, 5, 1], [2, 7, 6], [4, 3, 8]]) is False and is_magic_todo([[5] * 3] * 3) is False

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng cách **tra bảng 8 ma phương** (hãy tự dựng bằng đối xứng) hoặc kiểm tra tâm `5`:

In [34]:
class MySolution:
    def numMagicSquaresInside(self, grid):
        pass            # TODO


# Khi làm xong:
# assert MySolution().numMagicSquaresInside(ex1) == 1 and MySolution().numMagicSquaresInside([[8]]) == 0

### Bài 5 (mini challenge): ma phương `n × n`

Viết `numMagicSquaresNxN(grid, n)` đếm cửa sổ `n × n` là ma phương bậc `n` (chứa số phân biệt `1..n²`, mọi hàng/cột/hai chéo tổng `n(n²+1)/2`). Kiểm tra: với `n = 3` bằng bài gốc; với `n = 4` bằng ma phương Dürer đặt trong lưới.

In [35]:
def numMagicSquaresNxN(grid, n):
    pass            # TODO


# Khi làm xong:
# assert numMagicSquaresNxN(ex1, 3) == 1

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: đây là ma phương (một trong 8) nên mọi tổng `15`.
- Mức 2: bảng thứ hai đổi chỗ `9` và `2` ở hàng cuối, phá tổng.
- Mức 3: kết quả `1, 0`; lưới `5 × 7` có `(5−2)(7−2) = 15` cửa sổ.

**Bài 2**

- Mức 1: tính hàng `15, 15, 15`, cột `15, 15, 15`.
- Mức 2: chéo `6 + 5 + 4 = 15`, `8 + 5 + 2 = 15`; chín số là `1..9`.
- Mức 3: (a) ma phương; (b) `(4−2)(5−2) = 6`; (c) chín số có `9` hai lần và thiếu `8` (không phân biệt), tổng hàng cuối `4+3+9 = 16 ≠ 15`.

**Bài 3**

- Mức 1: `sorted(flat) == list(range(1, 10))`.
- Mức 2: `all(sum(r) == 15 for r in block)`.
- Mức 3: cột: `zip(*block)`; chéo: hai biểu thức.

**Bài 4**

- Mức 1: dựng 8 ma phương từ Lo Shu bằng `rotate90` và lật.
- Mức 2: mã hoá cửa sổ thành tuple 9 số.
- Mức 3: đếm cửa sổ có tuple ∈ tập.

**Bài 5**

- Mức 1: tổng mục tiêu `n(n²+1)/2`.
- Mức 2: kiểm tra `sorted(flat) == list(range(1, n*n+1))`.
- Mức 3: dùng `count_magic_n` ở mục 15.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert` để chạy kiểm tra.

### Bài 1

In [36]:
b = [[8, 1, 6], [3, 5, 7], [4, 9, 2]]
assert [sum(r) for r in b] == [15, 15, 15] and [sum(c) for c in zip(*b)] == [15, 15, 15]
assert (b[0][0] + b[1][1] + b[2][2], b[0][2] + b[1][1] + b[2][0]) == (15, 15)
assert Solution().numMagicSquaresInside(b) == 1 and Solution().numMagicSquaresInside([[8, 1, 6], [3, 5, 7], [4, 2, 9]]) == 0
assert len(list(windows([[0] * 7] * 5))) == 15
print("Bài 1 OK")

Bài 1 OK


### Bài 2

(a) `[[6,1,8],[7,5,3],[2,9,4]]`: hàng `15, 15, 15`; cột `15, 15, 15`; chéo `15, 15`; chín số `1..9` → ma phương. (b) `6`. (c) chín số có `9` hai lần và thiếu `8`; ngoài ra tổng hàng cuối `4 + 3 + 9 = 16 ≠ 15`.

In [37]:
x = [[6, 1, 8], [7, 5, 3], [2, 9, 4]]
assert is_magic_brute(x) and Solution().numMagicSquaresInside(x) == 1
assert len(list(windows([[0] * 5] * 4))) == 6
y = [[2, 7, 6], [9, 5, 1], [4, 3, 9]]
assert sorted(v for r in y for v in r) != list(range(1, 10)) and sum(y[2]) == 16
assert Solution().numMagicSquaresInside(y) == 0
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [38]:
def is_magic_answer(block):
    flat = [v for r in block for v in r]
    if sorted(flat) != list(range(1, 10)):
        return False
    return (all(sum(r) == 15 for r in block)
            and all(sum(c) == 15 for c in zip(*block))
            and block[0][0] + block[1][1] + block[2][2] == 15
            and block[0][2] + block[1][1] + block[2][0] == 15)


assert is_magic_answer([[4, 3, 8], [9, 5, 1], [2, 7, 6]]) is True
assert is_magic_answer([[9, 5, 1], [2, 7, 6], [4, 3, 8]]) is False and is_magic_answer([[5] * 3] * 3) is False
assert {p for p in permutations(range(1, 10)) if is_magic_answer([p[0:3], p[3:6], p[6:9]])} == MAGIC_SET
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [39]:
class MySolutionAnswer:
    def numMagicSquaresInside(self, grid):
        base = ((2, 7, 6), (9, 5, 1), (4, 3, 8))
        table, cur = set(), base
        for _ in range(4):
            table.add(tuple(v for r in cur for v in r))
            table.add(tuple(v for r in cur for v in r[::-1]))
            cur = tuple(zip(*cur[::-1]))
        m, n = len(grid), len(grid[0])
        return sum(tuple(grid[i + a][j + b] for a in range(3) for b in range(3)) in table
                   for i in range(m - 2) for j in range(n - 2))


assert MySolutionAnswer().numMagicSquaresInside(ex1) == 1 and MySolutionAnswer().numMagicSquaresInside([[8]]) == 0
rng = random.Random(4)
for _ in range(300):
    g = random_grid(rng, rng.randint(1, 8), rng.randint(1, 8), 9)
    if len(g) >= 3 and len(g[0]) >= 3 and rng.random() < 0.6:
        plant(g, rng.randint(0, len(g) - 3), rng.randint(0, len(g[0]) - 3), rng.choice(sq8))
    assert MySolutionAnswer().numMagicSquaresInside(g) == count_brute(g)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [40]:
def numMagicSquaresNxN_answer(grid, n):
    target = n * (n * n + 1) // 2
    full = list(range(1, n * n + 1))
    m, w = len(grid), len(grid[0])
    ans = 0
    for i in range(m - n + 1):
        for j in range(w - n + 1):
            block = [grid[i + a][j:j + n] for a in range(n)]
            if sorted(v for r in block for v in r) != full:
                continue
            if (all(sum(r) == target for r in block) and all(sum(c) == target for c in zip(*block))
                    and sum(block[k][k] for k in range(n)) == target and sum(block[k][n - 1 - k] for k in range(n)) == target):
                ans += 1
    return ans


assert numMagicSquaresNxN_answer(ex1, 3) == 1
assert numMagicSquaresNxN_answer(big, 4) == 1 and numMagicSquaresNxN_answer(durer, 4) == 1
assert numMagicSquaresNxN_answer([[1]], 1) == 1 and numMagicSquaresNxN_answer([[2]], 1) == 0
rng = random.Random(5)
for _ in range(200):
    g = random_grid(rng, rng.randint(3, 8), rng.randint(3, 8), 9)
    assert numMagicSquaresNxN_answer(g, 3) == Solution().numMagicSquaresInside(g)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (đếm ma phương `3 × 3`, tâm `5`):**

```python
class Solution:
    def numMagicSquaresInside(self, grid):
        m, n = len(grid), len(grid[0])
        ans = 0
        for i in range(m - 2):
            for j in range(n - 2):
                if grid[i + 1][j + 1] != 5:
                    continue
                a, b, c = grid[i][j:j + 3]
                d, e, f = grid[i + 1][j:j + 3]
                g, h, k = grid[i + 2][j:j + 3]
                if set((a, b, c, d, e, f, g, h, k)) == set(range(1, 10)) \
                        and a + k == c + g == b + h == d + f == 10 \
                        and a + b + c == g + h + k == a + d + g == c + f + k == 15:
                    ans += 1
        return ans
```

**Template (tra bảng):** dựng 8 ma phương từ `((2,7,6),(9,5,1),(4,3,8))` bằng 4 phép xoay × (không lật / lật), mã hoá cửa sổ thành tuple 9 số, đếm cửa sổ ∈ tập.

**Checklist trước khi submit:**

- [ ] Chín số là **hoán vị của `1..9`** (phân biệt và trong khoảng)?
- [ ] Đủ **8 đường**: 3 hàng, 3 cột, **cả hai** đường chéo?
- [ ] Nhớ rằng "tâm `5`" chỉ là **điều kiện cần**, không đủ?
- [ ] `range(m - 2)`, `range(n - 2)` (không phải `m - 3`); lưới nhỏ hơn 3 cho `0`?
- [ ] Đã thử toàn `5`, ma phương bán phần (đổi hai hàng), ma phương `+1`, lưới `[[8]]`?
- [ ] Nếu tra bảng: bảng đủ **8** phần tử (đã kiểm bằng đối xứng hoặc vét cạn)?
- [ ] Đúng tên `class Solution`, `numMagicSquaresInside(self, grid)`, không in debug.